### CA2 - Credit Default Risk: Exploratory Data Analysis and Data Preparation
**Module:** Strategic Thinking                                   
**Programme:** Higher Diploma in Data Analytics in Business                            
**Student:** Dashzeveg Purevtogtokh  
**Student ID:** 2026328                         
**Assignment:** CA2                                     
**GitHub Repository:** https://github.com/2026328-Dashzeveg-Purevtogtokh/Strategic-Thinking-Sep26

This notebook continues the CA1 capstone project.  
The main purpose of this CA2 notebook is to:

- understand the credit-card default dataset;
- check data quality;
- perform descriptive statistics;
- perform exploratory data analysis (EDA);
- apply suitable inferential statistics;
- prepare the data for later machine-learning work;
- record findings, limitations and recommendations.

#### 1. Import Libraries

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from scipy.stats import chi2_contingency, ttest_ind

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

#### 2. Load the Dataset

In [11]:
from pathlib import Path
import pandas as pd

filename = "default of credit card clients.xls"

paths = [
    Path("data") / filename,
    Path("CA1-Credit-Default-Risk/data") / filename,
]

DATA_PATH = next((p for p in paths if p.is_file()), None)

if DATA_PATH is None:
    raise FileNotFoundError(f"Dataset not found from: {Path.cwd()}")

df_raw = pd.read_excel(DATA_PATH, header=1, engine="calamine")
df = df_raw.copy()

print("Loaded:", DATA_PATH)
print("Rows, columns:", df_raw.shape)
display(df_raw.head())

Loaded: data\default of credit card clients.xls
Rows, columns: (30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0


#### 3. Initial Data Understanding

The CA1 project identified this as the UCI **Default of Credit Card Clients** dataset.

The dataset contains historical customer credit and repayment information.  
Important groups of variables include:

- `LIMIT_BAL` – credit limit;
- `SEX`, `EDUCATION`, `MARRIAGE`, `AGE` – customer characteristics;
- `PAY_0` to `PAY_6` – repayment-status variables;
- `BILL_AMT1` to `BILL_AMT6` – bill amounts;
- `PAY_AMT1` to `PAY_AMT6` – previous payment amounts;
- `default payment next month` – target variable.

The next cells confirm the structure of the actual uploaded dataset.

In [12]:
df = df.replace(["none", "None", "NONE", "null", "NULL", "NaN", ""], np.nan)

In [13]:
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [14]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")

Number of rows: 30000
Number of columns: 25

Column names:
01. ID
02. LIMIT_BAL
03. SEX
04. EDUCATION
05. MARRIAGE
06. AGE
07. PAY_0
08. PAY_2
09. PAY_3
10. PAY_4
11. PAY_5
12. PAY_6
13. BILL_AMT1
14. BILL_AMT2
15. BILL_AMT3
16. BILL_AMT4
17. BILL_AMT5
18. BILL_AMT6
19. PAY_AMT1
20. PAY_AMT2
21. PAY_AMT3
22. PAY_AMT4
23. PAY_AMT5
24. PAY_AMT6
25. default payment next month


In [15]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
print(df.columns.tolist())

['id', 'limit_bal', 'sex', 'education', 'marriage', 'age', 'pay_0', 'pay_2', 'pay_3', 'pay_4', 'pay_5', 'pay_6', 'bill_amt1', 'bill_amt2', 'bill_amt3', 'bill_amt4', 'bill_amt5', 'bill_amt6', 'pay_amt1', 'pay_amt2', 'pay_amt3', 'pay_amt4', 'pay_amt5', 'pay_amt6', 'default_payment_next_month']


In [16]:
# Rename target value
df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]
df = df.rename(columns={"default_payment_next_month": "default"})

In [17]:
df = df.map(lambda s: s.lower() if isinstance(s, str) else s)

In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   id         30000 non-null  int64
 1   limit_bal  30000 non-null  int64
 2   sex        30000 non-null  int64
 3   education  30000 non-null  int64
 4   marriage   30000 non-null  int64
 5   age        30000 non-null  int64
 6   pay_0      30000 non-null  int64
 7   pay_2      30000 non-null  int64
 8   pay_3      30000 non-null  int64
 9   pay_4      30000 non-null  int64
 10  pay_5      30000 non-null  int64
 11  pay_6      30000 non-null  int64
 12  bill_amt1  30000 non-null  int64
 13  bill_amt2  30000 non-null  int64
 14  bill_amt3  30000 non-null  int64
 15  bill_amt4  30000 non-null  int64
 16  bill_amt5  30000 non-null  int64
 17  bill_amt6  30000 non-null  int64
 18  pay_amt1   30000 non-null  int64
 19  pay_amt2   30000 non-null  int64
 20  pay_amt3   30000 non-null  int64
 21  pay_amt4   30000 non-nu

**My observation:** The dataset contains 30,000 records and 25 columns. All expected variables are present, including customer information, repayment status, bill amounts and previous payment amounts. The original target column, `default payment next month`, was renamed to `default` to make it easier to use in the analysis. All columns are currently stored as integer (`int64`) data types. No unexpected data types were identified. The `id` column is a customer identifier and will not be used as a predictive feature in the later modelling stage.

#### 4. Data Quality Checks
Before analysing patterns, check:
1. missing values;
2. duplicate rows;
3. unique values in important categorical variables;
4. unusual or invalid category codes;
5. possible extreme numerical values.

In [20]:
# Missing values
df.isnull().sum()

id           0
limit_bal    0
sex          0
education    0
marriage     0
age          0
pay_0        0
pay_2        0
pay_3        0
pay_4        0
pay_5        0
pay_6        0
bill_amt1    0
bill_amt2    0
bill_amt3    0
bill_amt4    0
bill_amt5    0
bill_amt6    0
pay_amt1     0
pay_amt2     0
pay_amt3     0
pay_amt4     0
pay_amt5     0
pay_amt6     0
default      0
dtype: int64

In [21]:
# Duplicate rows
df.duplicated().sum()

np.int64(0)

### Categorical Variable Codes
The dataset uses numerical codes for several categorical variables.  
The main meanings of these codes are shown below.
#### SEX
| Code | Meaning |
|---:|---|
| 1 | Male |
| 2 | Female |

#### EDUCATION
| Code | Meaning |
|---:|---|
| 1 | Graduate school |
| 2 | University |
| 3 | High school |
| 4 | Others |

> **Note:** The dataset also contains values such as `0`, `5`, and `6`.  

#### MARRIAGE
| Code | Meaning |
|---:|---|
| 1 | Married |
| 2 | Single |
| 3 | Others |

#### DEFAULT
| Code | Meaning |
|---:|---|
| 0 | No default |
| 1 | Default |

The `default` variable is the **target variable** of this project.  
A value of `1` means that the customer defaulted on the credit card payment in the following month, while `0` means that the customer did not default.

In [29]:
for column in ["sex", "education", "marriage", "default"]:
    print(f"\n{column}")
    display(df[column].value_counts(dropna=False).sort_index().to_frame("Count"))


sex


,Count
sex,
1,11888
2,18112



education


,Count
education,
0,14
1,10585
2,14030
3,4917
4,123
5,280
6,51



marriage


,Count
marriage,
0,54
1,13659
2,15964
3,323



default


,Count
default,
0,23364
1,6636


#### Repayment Status Variables

The variables `PAY_0`, `PAY_2`, `PAY_3`, `PAY_4`, `PAY_5`, and `PAY_6` represent the customer's repayment status over the previous six months.

| Variable | Month represented |
|---|---|
| `PAY_0` | September 2005 |
| `PAY_2` | August 2005 |
| `PAY_3` | July 2005 |
| `PAY_4` | June 2005 |
| `PAY_5` | May 2005 |
| `PAY_6` | April 2005 |

#### Repayment Status Codes

| Code | Meaning |
|---:|---|
| `-2` | No consumption |
| `-1` | Payment made on time |
| `0` | Revolving credit used |
| `1` | Payment delayed by 1 month |
| `2` | Payment delayed by 2 months |
| `3` | Payment delayed by 3 months |
| `4` | Payment delayed by 4 months |
| `5` | Payment delayed by 5 months |
| `6` | Payment delayed by 6 months |
| `7` | Payment delayed by 7 months |
| `8` | Payment delayed by 8 months |
| `9` | Payment delayed by 9 months or more |

> **Note:** The dataset also contains values such as `-2` and `0`. The official UCI description does not clearly define the values `-2` and `0`. However, several secondary dataset descriptions interpret -2 as no consumption and 0 as the use of revolving credit. Therefore, these values are treated as meaningful repayment-status categories rather than data errors.

These repayment variables are important for this project because they describe the customer's previous payment behaviour. A higher positive value represents a longer payment delay and may be associated with a higher risk of default.

In [30]:
for column in ["pay_0", "pay_2", "pay_3", "pay_4", "pay_5", "pay_6"]:
    print(f"\n{column}")
    display(df[column].value_counts(dropna=False).sort_index().to_frame("Count"))


pay_0


,Count
pay_0,
-2,2759
-1,5686
0,14737
1,3688
2,2667
3,322
4,76
5,26
6,11



pay_2


,Count
pay_2,
-2,3782
-1,6050
0,15730
1,28
2,3927
3,326
4,99
5,25
6,12



pay_3


,Count
pay_3,
-2,4085
-1,5938
0,15764
1,4
2,3819
3,240
4,76
5,21
6,23


... (2 outputs left out)


pay_5


,Count
pay_5,
-2,4546
-1,5539
0,16947
2,2626
3,178
4,84
5,17
6,4
7,58



pay_6


,Count
pay_6,
-2,4895
-1,5740
0,16286
2,2766
3,184
4,49
5,13
6,19
7,46


**Interpretation:** No missing values or exact duplicate records were found in the dataset. However, the `education` variable contains some unexpected category codes such as 0, 5 and 6, while the `marriage` variable also contains the value 0. These values need to be checked against the dataset documentation before they are grouped or recoded.
The repayment-status variables also contain negative values such as -1 and -2, as well as 0. These values should not be treated as errors or removed automatically because they are part of the original dataset coding. They will be investigated further and, if necessary, grouped during the data-preparation stage.

In [31]:
df.groupby("pay_0")["default"].agg(["count", "mean"])

,count,mean
pay_0,,
-2,2759,0.13
-1,5686,0.17
0,14737,0.13
1,3688,0.34
2,2667,0.69
3,322,0.76
4,76,0.68
5,26,0.50
6,11,0.55


#### 5. Descriptive Statistics
Descriptive statistics help explain the main characteristics of the dataset before modelling.

In [26]:
# Summary statistics for numerical variables
df.describe().T

,count,mean,std,min,25%,50%,75%,max
id,"30,000.00","15,000.50","8,660.40",1.00,"7,500.75","15,000.50","22,500.25","30,000.00"
limit_bal,"30,000.00","167,484.32","129,747.66","10,000.00","50,000.00","140,000.00","240,000.00","1,000,000.00"
sex,"30,000.00",1.60,0.49,1.00,1.00,2.00,2.00,2.00
education,"30,000.00",1.85,0.79,0.00,1.00,2.00,2.00,6.00
marriage,"30,000.00",1.55,0.52,0.00,1.00,2.00,2.00,3.00
age,"30,000.00",35.49,9.22,21.00,28.00,34.00,41.00,79.00
pay_0,"30,000.00",-0.02,1.12,-2.00,-1.00,0.00,0.00,8.00
pay_2,"30,000.00",-0.13,1.20,-2.00,-1.00,0.00,0.00,8.00
pay_3,"30,000.00",-0.17,1.20,-2.00,-1.00,0.00,0.00,8.00
pay_4,"30,000.00",-0.22,1.17,-2.00,-1.00,0.00,0.00,8.00
